In [22]:
import numpy as np
import os
import data_ingestion_
from data_preparation_ import prepare_data_for_train_val_test_sets_, EmoDataset

the_project_info_ = data_ingestion_.get_toml_info_()
goemotions_ml_loc_ = the_project_info_["dataset_goemotions"]["goemotions_mldata_location_"]
train_ds_ = EmoDataset(goemotions_ml_loc_+"train.txt")
print(f"The text is -->> {train_ds_[200][0]},  and corresponding label is {train_ds_[200][1]}" )

The text is -->> Dirt bar downtown is pretty sweet.,  and corresponding label is 27


In [30]:
# -- pre trained model for emotions classification
from tokenizer_fn_ import pretrained_model_emoClassification, tokenize_function
the_text_ = train_ds_[200][0]
class_ = pretrained_model_emoClassification(the_text_)
print(class_)
tokens_ = tokenize_function(the_text_)
print(np.shape(tokens_['input_ids']))
print(np.shape(tokens_['attention_mask']))

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 5380.89it/s]

[{'label': 'POSITIVE', 'score': 0.9998219609260559}]
(128,)
(128,)


In [20]:
from typing import Dict
import numpy as np
import ray

def add_dog_years(batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    batch["age_in_dog_years"] = 7 * batch["age"]
    return batch

ds = (
    ray.data.from_items([
        {"name": "Luna", "age": 4},
        {"name": "Rory", "age": 14},
        {"name": "Scout", "age": 9},
    ])
    .map_batches(add_dog_years)
)
ds.take_batch()

# tokenized_datasets = train_ds_.map(tokenize_function, batched=True)
# tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

2026-03-27 00:09:38,270	INFO logging.py:392 -- Registered dataset logger for dataset dataset_5_0
2026-03-27 00:09:38,271	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_5_0. Full logs are in C:\Users\RAVISH~1\AppData\Local\Temp\ray\session_2026-03-27_00-09-10_935501_28836\logs\ray-data
2026-03-27 00:09:38,276	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_5_0: InputDataBuffer[Input] -> TaskPoolMapOperator[MapBatches(add_dog_years)] -> LimitOperator[limit=20]
2026-03-27 00:09:38,384	INFO logging_progress.py:174 -- ======= Running Dataset: dataset_5_0 =======
2026-03-27 00:09:38,388	INFO logging_progress.py:225 -- Total Progress: 0/?
2026-03-27 00:09:38,389	INFO logging_progress.py:227 -- Active & requested resources: 0/8 CPU, 0.0B/295.9MiB object store
2026-03-27 00:09:38,391	INFO logging_progress.py:181 -- 
2026-03-27 00:09:38,392	INFO logging_progress.py:231 -- MapBatches(add_dog_years): 0/1
2026-03-27 00:09:38,393	INFO logging_progress.py:

{'name': array(['Luna', 'Rory', 'Scout'], dtype=object),
 'age': array([ 4, 14,  9]),
 'age_in_dog_years': array([28, 98, 63])}

In [34]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = model.base_model

classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

enc = tokenizer(the_text_)
token_representations = base_model(torch.tensor(enc["input_ids"]).unsqueeze(0))[0][0]

print(enc["input_ids"])
print(tokenizer.decode(enc["input_ids"]))
print(f"Length: {len(enc['input_ids'])}")
print(token_representations.shape)



Loading weights: 100%|██████████| 104/104 [00:00<00:00, 10514.07it/s]


[101, 6900, 3347, 5116, 2003, 3492, 4086, 1012, 102]
[CLS] dirt bar downtown is pretty sweet. [SEP]
Length: 9
torch.Size([9, 768])


In [53]:
from model_definition_ import EmoModel
from transformers import AutoModelForMaskedLM
X = torch.tensor(enc["input_ids"]).unsqueeze(0).to('cpu')
attn = torch.tensor(enc["attention_mask"]).unsqueeze(0).to('cpu')
print(X.shape, attn.shape)
classifier_emomodel_ = EmoModel(AutoModelForMaskedLM.from_pretrained("distilroberta-base").base_model, 20)
hidd_ = classifier_emomodel_((X, attn))
print(hidd_)
print(hidd_.shape)

torch.Size([1, 9]) torch.Size([1, 9])


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 5870.73it/s]
RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[-0.0358, -0.2659,  0.2187, -0.0799, -0.0539, -0.0167, -0.1176,  0.0194,
          0.1442,  0.1175, -0.0163,  0.0682,  0.0916,  0.2916, -0.2231,  0.1395,
          0.0076, -0.1508, -0.1654,  0.1441]], grad_fn=<AddmmBackward0>)
torch.Size([1, 20])


In [66]:
dataframe_emotions_, label_emotions_ = data_ingestion_.prepare_dataframe(the_project_info_)
print(label_emotions_)
desired_emotion_lab_ = label_emotions_

['admiration' 'amusement' 'anger' 'annoyance' 'approval' 'caring'
 'confusion' 'curiosity' 'desire' 'disappointment' 'disapproval' 'disgust'
 'embarrassment' 'excitement' 'fear' 'gratitude' 'grief' 'joy' 'love'
 'nervousness' 'optimism' 'pride' 'realization' 'relief' 'remorse'
 'sadness' 'surprise' 'neutral']


In [59]:
from tokenizers import ByteLevelBPETokenizer
from tokenizers.processors import BertProcessing
class TokenizersCollateFn:
    def __init__(self, max_tokens=512):
      ## RoBERTa uses BPE tokenizer similar to GPT
      t = ByteLevelBPETokenizer(
          "tokenizer/vocab.json",
          "tokenizer/merges.txt"
      )
      t._tokenizer.post_processor = BertProcessing(
          ("</s>", t.token_to_id("</s>")),
          ("<s>", t.token_to_id("<s>")),
      )
      t.enable_truncation(max_tokens)
      t.enable_padding(length=max_tokens, pad_id=t.token_to_id("<pad>"))
      self.tokenizer = t

    def __call__(self, batch):
      encoded = self.tokenizer.encode_batch([x[0] for x in batch])
      sequences_padded = torch.tensor([enc.ids for enc in encoded])
      attention_masks_padded = torch.tensor([enc.attention_mask for enc in encoded])
      labels = torch.tensor([x[1] for x in batch])

      return (sequences_padded, attention_masks_padded), labels

In [71]:
## Methods required by PyTorchLightning
import pytorch_lightning as pl
from torch import nn
from typing import List
from functools import lru_cache
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW as AdamW
from transformers import get_linear_schedule_with_warmup


class TrainingModule(pl.LightningModule):
    def __init__(self, hparams):
        super().__init__()
        self.model = EmoModel(AutoModelForSequenceClassification.from_pretrained("distilroberta-base").base_model, len(desired_emotion_lab_))
        self.loss = nn.CrossEntropyLoss() ## combines LogSoftmax() and NLLLoss()
        #self.hparams = hparams
        self.hparams.update(vars(hparams))

    def step(self, batch, step_name="train"):
        X, y = batch
        loss = self.loss(self.forward(X), y)
        loss_key = f"{step_name}_loss"
        tensorboard_logs = {loss_key: loss}

        return { ("loss" if step_name == "train" else loss_key): loss, 'log': tensorboard_logs,
               "progress_bar": {loss_key: loss}}

    def forward(self, X, *args):
        return self.model(X, *args)

    def training_step(self, batch, batch_idx):
        return self.step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self.step(batch, "val")

    def validation_end(self, outputs: List[dict]):
        loss = torch.stack([x["val_loss"] for x in outputs]).mean()
        return {"val_loss": loss}

    def test_step(self, batch, batch_idx):
        return self.step(batch, "test")

    def train_dataloader(self):
        return self.create_data_loader(self.hparams.train_path, shuffle=True)

    def val_dataloader(self):
        return self.create_data_loader(self.hparams.val_path)

    def test_dataloader(self):
        return self.create_data_loader(self.hparams.test_path)

    def create_data_loader(self, ds_path: str, shuffle=False):
        return DataLoader(
                    EmoDataset(ds_path),
                    batch_size=self.hparams.batch_size,
                    shuffle=shuffle,
                    collate_fn=TokenizersCollateFn()
        )

    @lru_cache()
    def total_steps(self):
        return len(self.train_dataloader()) // self.hparams.accumulate_grad_batches * self.hparams.epochs

    def configure_optimizers(self):
        ## use AdamW optimizer -- faster approach to training NNs
        ## read: https://www.fast.ai/2018/07/02/adam-weight-decay/
        optimizer = AdamW(self.model.parameters(), lr=self.hparams.lr)
        lr_scheduler = get_linear_schedule_with_warmup(
                    optimizer,
                    num_warmup_steps=self.hparams.warmup_steps,
                    num_training_steps=self.total_steps(),
        )
        return [optimizer], [{"scheduler": lr_scheduler, "interval": "step"}]

In [76]:

from argparse import Namespace
goemotions_ml_loc_ = the_project_info_["dataset_goemotions"]["goemotions_mldata_location_"]
print(goemotions_ml_loc_)
train_path =  goemotions_ml_loc_ + 'train.txt'
val_path =  goemotions_ml_loc_ + 'val.txt'
test_path =  goemotions_ml_loc_ + 'test.txt'

lr=0.1 ## uper bound LR
from torch_lr_finder import LRFinder

hparams_tmp = Namespace(
    train_path=train_path,
    val_path=val_path,
    test_path=test_path,
    batch_size=16,
    warmup_steps=100,
    epochs=1,
    lr=lr,
    accumulate_grad_batches=1,
)
module = TrainingModule(hparams_tmp)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(module.parameters(), lr=5e-7) ## lower bound LR
lr_finder = LRFinder(module, optimizer, criterion, device="cuda")
lr_finder.range_test(module.train_dataloader(), end_lr=100, num_iter=100, accumulation_steps=hparams_tmp.accumulate_grad_batches)
lr_finder.plot()
lr_finder.reset()

data_goemotions_/data/train_val_test_sets_/content/


Loading weights: 100%|██████████| 101/101 [00:00<00:00, 1295.40it/s]
RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Exception: Error while initializing BPE: The system cannot find the path specified. (os error 3)

In [ ]:
from torch.utils.data import DataLoader
from torch_lr_finder import LRFinder

desired_batch_size, real_batch_size = 32, 4
accumulation_steps = desired_batch_size // real_batch_size

dataset = train_path
real_batch_size = 50
# Beware of the `batch_size` used by `DataLoader`
trainloader = DataLoader(dataset, batch_size=real_batch_size, shuffle=True)
# classifier_emomodel_ = EmoModel(AutoModelForMaskedLM.from_pretrained("distilroberta-base").base_model, len(desired_emotion_lab_))

model = classifier_emomodel_
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(module.parameters(), lr=5e-7) ## lower bound LR

# (Optional) With this setting, `amp.scale_loss()` will be adopted automatically.
# model, optimizer = amp.initialize(model, optimizer, opt_level='O1')

lr_finder = LRFinder(model, optimizer, criterion, device="cuda")
lr_finder.range_test(trainloader, end_lr=10, num_iter=100, step_mode="exp", accumulation_steps=accumulation_steps)
lr_finder.plot()
lr_finder.reset()

TypeError: object of type 'ellipsis' has no len()